# 03 - Verify infrastructure and metadata readiness

Run from the notebook directory after the deployment/export completes. These read-only SDK checks verify the current environment and same CLI user; they do not create an index, upload documents, or execute search queries.

An underlying access/policy error blocks readiness. Resolve the prerequisite and explicitly rerun; do not enable keys, switch identity, or introduce fallback paths. Old evidence is historical until this run completes.

## Step 1 - Validate the handoff against the current CLI user

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

from azure.identity import AzureCliCredential
from azure.mgmt.resource.resources import ResourceManagementClient
from azure.mgmt.resource.deployments import DeploymentsMgmtClient
from azure.search.documents.indexes import SearchIndexClient
from dotenv import dotenv_values

demo_root = Path('..').resolve()
sys.path.insert(0, str(demo_root))
from app.notebook_helpers import environment_from_outputs, resolve_az_cli, validate_environment

RESOURCE_GROUP = "rg-search02-hybrid-pagination-ne"
DEMO_ID = 'search02-hybrid-pagination'
DEPLOYMENT_NAME = 'search02-hybrid-pagination'
SEARCH_API_VERSION = '2025-09-01'
generated = demo_root / 'outputs' / 'generated'
az_cmd = resolve_az_cli()
if az_cmd is None:
    raise RuntimeError('Azure CLI not found. Complete setup first.')
result = subprocess.run([az_cmd, 'account', 'show', '--output', 'json'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f'az account show failed ({result.returncode})\nstderr: {result.stderr.strip()}\nstdout: {result.stdout.strip()}')
account = json.loads(result.stdout)
subscription_id = os.environ.get('AZURE_SUBSCRIPTION_ID', account['id'])
if subscription_id.casefold() != account['id'].casefold() or account['user']['type'].casefold() != 'user':
    raise ValueError('Select the intended subscription and sign in as the setup user before validation.')
tenant_id = account['tenantId']
result = subprocess.run([az_cmd, 'ad', 'signed-in-user', 'show', '--query', 'id', '--output', 'tsv'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f'az ad signed-in-user show failed ({result.returncode})\nstderr: {result.stderr.strip()}\nstdout: {result.stdout.strip()}')
operator_object_id = result.stdout.strip()
environment = validate_environment(dotenv_values(demo_root / 'env' / '.env'), subscription_id=subscription_id, tenant_id=tenant_id, resource_group=RESOURCE_GROUP, operator_object_id=operator_object_id)
credential = AzureCliCredential(subscription=subscription_id)
resource_client = ResourceManagementClient(credential, subscription_id)
deployments_client = DeploymentsMgmtClient(credential, subscription_id)
print(f'Validated non-secret context for user {operator_object_id} in {subscription_id}.')

## Step 2 - Match the actual successful deployment and owned resources

In [ ]:
deployment = deployments_client.deployments.get_at_subscription_scope(DEPLOYMENT_NAME)
if deployment.properties.provisioning_state != 'Succeeded':
    raise RuntimeError(f'Latest deployment is not successful: {deployment.properties.provisioning_state}')
current_environment = environment_from_outputs(deployment.properties.outputs, subscription_id=subscription_id, tenant_id=tenant_id, resource_group=RESOURCE_GROUP, operator_object_id=operator_object_id)
if current_environment != environment:
    raise ValueError('Export is stale; rerun the successful deployment output-export step.')
group = resource_client.resource_groups.get(RESOURCE_GROUP)
if group.tags is None or group.tags.get('demo') != DEMO_ID:
    raise ValueError('Resource group ownership does not match this demo.')
service = resource_client.resources.get_by_id(environment['AZURE_SEARCH_RESOURCE_ID'], api_version='2025-05-01')
properties = service.properties
if service.sku.name != 'basic' or service.location.replace(' ', '').casefold() != 'northeurope':
    raise ValueError('Live tier/location does not match approved Bicep settings.')
if properties['replicaCount'] != 1 or properties['partitionCount'] != 1:
    raise ValueError('Live capacity does not match one replica and one partition.')
if properties['disableLocalAuth'] is not True or properties['publicNetworkAccess'].casefold() != 'enabled' or properties['networkRuleSet']['ipRules'] != []:
    raise ValueError('Live authentication/networking does not match the approved public, keyless design.')
print('Actual deployment outputs, group ownership, capacity, and public/keyless settings match.')

## Step 3 - Inspect the two Bicep-owned role assignments

Use the nested deployment's resource IDs, not inherited permissions, to inspect the exact assignments. This proves assigned roles, not document-operation execution.

In [ ]:
module_deployment = deployments_client.deployments.get(RESOURCE_GROUP, f'{DEMO_ID}-search')
assignment_prefix = environment['AZURE_SEARCH_RESOURCE_ID'] + '/providers/Microsoft.Authorization/roleAssignments/'
assignment_ids = sorted(resource.id for resource in module_deployment.properties.output_resources if resource.id.casefold().startswith(assignment_prefix.casefold()))
if len(assignment_ids) != 2 or len(set(assignment_ids)) != 2:
    raise ValueError('The Bicep module must provision exactly two distinct service-scoped assignments.')
expected_roles = {'7ca78c08-252a-4471-8644-bb5ff32d4ba0', '8ebe5a00-799e-43f5-93ac-243d3dce84a7'}
seen_roles = set()
role_evidence = []
for assignment_id in assignment_ids:
    assignment = resource_client.resources.get_by_id(assignment_id, api_version='2022-04-01')
    role = assignment.properties
    role_id = role['roleDefinitionId'].rsplit('/', 1)[1].casefold()
    if role['principalId'].casefold() != operator_object_id.casefold() or role['principalType'] != 'User' or role_id not in expected_roles or role_id in seen_roles:
        raise ValueError(f'Unexpected role or principal on assignment {assignment_id}')
    seen_roles.add(role_id)
    role_evidence.append({'id': assignment.id, 'principalId': role['principalId'], 'principalType': role['principalType'], 'roleDefinitionId': role['roleDefinitionId'], 'scope': environment['AZURE_SEARCH_RESOURCE_ID']})
if seen_roles != expected_roles:
    raise ValueError('The required index-management and document/query roles are incomplete.')
print(json.dumps(role_evidence, indent=2))

## Step 4 - Read service statistics through the stable Python SDK

The SDK obtains the same user's Search token for `https://search.azure.com/.default`. This initial Feature 1 check requires zero indexes/documents. If Feature 2 has already populated the service, do not delete its data to force this baseline to pass.

In [ ]:
with SearchIndexClient(environment['AZURE_SEARCH_ENDPOINT'], credential, api_version=SEARCH_API_VERSION) as index_client:
    statistics = index_client.get_service_statistics()
index_count = statistics.counters.index_counter.usage
document_count = statistics.counters.document_counter.usage
if type(index_count) is not int or type(document_count) is not int or index_count != 0 or document_count != 0:
    raise ValueError(f'Fresh-environment baseline requires zero indexes/documents, got {index_count}/{document_count}.')
print(f'Authorized metadata access succeeded: {index_count} indexes, {document_count} documents.')

## Step 5 - Record timestamped readiness

Write success only after all checks above pass. A raw failure remains a failed/blocked check, never a ready environment. Preserve its error separately if a prerequisite blocks this run.

In [ ]:
policy_review = json.loads((generated / 'policy-review.json').read_text(encoding='utf-8'))
if policy_review['subscriptionId'].casefold() != subscription_id.casefold() or policy_review['scope'].casefold() != environment['AZURE_RESOURCE_GROUP_ID'].casefold() or policy_review['decision'] != 'approved-public-keyless':
    raise ValueError('Policy evidence does not match this environment and approved design.')
readiness = {'status': 'ready', 'observedAt': datetime.now(timezone.utc).isoformat(), 'environment': environment, 'provisioningState': deployment.properties.provisioning_state, 'settings': {'sku': service.sku.name, 'location': service.location, 'replicas': properties['replicaCount'], 'partitions': properties['partitionCount'], 'disableLocalAuth': properties['disableLocalAuth'], 'publicNetworkAccess': properties['publicNetworkAccess'], 'ipRules': properties['networkRuleSet']['ipRules']}, 'roleAssignments': role_evidence, 'metadataStatus': 'passed', 'counters': {'indexes': index_count, 'documents': document_count}, 'sdkVersion': version('azure-search-documents'), 'apiVersion': SEARCH_API_VERSION, 'policyReview': policy_review, 'documentOperationStatus': 'deferred-to-feature-2'}
(generated / 'readiness.json').write_text(json.dumps(readiness, indent=2), encoding='utf-8')
print(json.dumps(readiness, indent=2))

## Next steps

Repeat the approved deployment with unchanged inputs and compare group/service/assignment IDs before cleanup acceptance. Infrastructure readiness does not establish the customer's pagination symptom.

When finished, follow [the cleanup guide](../README.md#cleanup) and obtain separate deletion approval. The canonical destroy notebook has confirmation flags already set to True; never use unattended Run All. Verified cleanup leaves `deleted`, makes this evidence historical and the export stale, and does not automatically authorize recreation.